In [9]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

REPO = Path("/Users/mkrishan/Documents/boolearn-main")
assert REPO.exists(), REPO
print("REPO:", REPO)

# >>> EDIT THIS ONLY if needed: where m5k1.dat..m5k4.dat already are <<<
MK_DIR = REPO  # <-- if m5k*.dat are in the repo root
# MK_DIR = REPO / "kaggle_random_majority"  # example if they are in a subfolder

# sanity check
for i in [1,2,3,4]:
    p = MK_DIR / f"m5k{i}.dat"
    print(p, "exists?", p.exists())

OUT = REPO / "paper_runs_adamw_vs_penalty"
OUT.mkdir(parents=True, exist_ok=True)
print("OUT:", OUT)

device: cpu
REPO: /Users/mkrishan/Documents/boolearn-main
/Users/mkrishan/Documents/boolearn-main/m5k1.dat exists? True
/Users/mkrishan/Documents/boolearn-main/m5k2.dat exists? True
/Users/mkrishan/Documents/boolearn-main/m5k3.dat exists? True
/Users/mkrishan/Documents/boolearn-main/m5k4.dat exists? True
OUT: /Users/mkrishan/Documents/boolearn-main/paper_runs_adamw_vs_penalty


In [11]:
def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    it = iter(lines)
    N_total = int(next(it))
    Tin, IN = map(int, next(it).split())
    Tout, OUTD = map(int, next(it).split())
    X_list, Y_list = [], []
    for _ in range(N_total):
        X_list.append(list(map(int, next(it).split())))
        Y_list.append(list(map(int, next(it).split())))
    X = torch.tensor(X_list, dtype=torch.float32)
    Y = torch.tensor(Y_list, dtype=torch.float32)
    return X, Y, IN, OUTD

def log_spaced_steps(max_steps: int, n_points: int = 45):
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return set([0] + steps.tolist())

In [13]:
loss_fn = nn.BCEWithLogitsLoss()

@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    bit_acc = (pred == y_true).float().mean().item() * 100.0
    exact_acc = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit_acc, exact_acc

def norm_penalty_rows(W: torch.Tensor, target_norm2: float):
    row_norm2 = (W * W).sum(dim=1)
    return ((row_norm2 - target_norm2) ** 2).mean()

def l1_penalty(W: torch.Tensor):
    return W.abs().mean()

In [15]:
class MLP32_depth5(nn.Module):
    def __init__(self):
        super().__init__()
        hidden = 32
        depth = 5
        layers = [nn.Linear(32, hidden), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Linear(hidden, hidden), nn.ReLU(inplace=True)]
        layers += [nn.Linear(hidden, 32)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class MLP16_32_32_16(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 16),
        )
    def forward(self, x): return self.net(x)

In [17]:
def run_adamw_and_penalty(
    task_name: str,
    X_all: torch.Tensor,
    Y_all: torch.Tensor,
    make_model_fn,
    N_train: int,
    max_steps: int,
    results_dir: Path,
    seed: int = 0,
    lr: float = 1e-3,
    wd: float = 1e-4,
    lam_norm: float = 1e-2,
    lam_l1: float = 1e-4,
    log_points: int = 45,
):
    results_dir.mkdir(parents=True, exist_ok=True)
    LOG_STEPS = log_spaced_steps(max_steps, log_points)
    torch.manual_seed(seed)

    Xtr = X_all[:N_train].to(device); Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device); Yte = Y_all[N_train:].to(device)

    def train(mode: str):
        model = make_model_fn().to(device)
        opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

        tag = f"{task_name}_{mode}_N{N_train}_seed{seed}_steps{max_steps}"
        log_path = results_dir / f"{tag}.log"

        with open(log_path, "w") as f:
            f.write("# step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

        xs, te_bits = [], []

        @torch.no_grad()
        def eval_log(step: int):
            model.eval()
            tr_logits = model(Xtr); te_logits = model(Xte)
            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()
            tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr)
            te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte)

            with open(log_path, "a") as f:
                f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_ex:.4f},{te_ex:.4f}\n")

            print(f"[{task_name} | {mode} | N={N_train}] step={step:>8d} "
                  f"train={tr_bit:6.2f}% test={te_bit:6.2f}% exact={tr_ex:5.1f}/{te_ex:5.1f}")

            xs.append(step); te_bits.append(te_bit)
            model.train()

        eval_log(0)
        model.train()

        for step in range(1, max_steps + 1):
            opt.zero_grad(set_to_none=True)
            logits = model(Xtr)
            base = loss_fn(logits, Ytr)

            if mode == "PenaltyAdamW":
                pen = 0.0
                for m in model.modules():
                    if isinstance(m, nn.Linear):
                        fan_in = m.weight.shape[1]
                        pen += lam_norm * norm_penalty_rows(m.weight, float(fan_in))
                        pen += lam_l1 * l1_penalty(m.weight)
                loss = base + pen
            else:
                loss = base

            loss.backward()
            opt.step()

            if step in LOG_STEPS:
                eval_log(step)

        return np.array(xs), np.array(te_bits), log_path

    xsA, teA, logA = train("AdamW")
    xsP, teP, logP = train("PenaltyAdamW")

    # Save curve
    fig = plt.figure()
    plt.plot(xsA, teA, marker="o", label="AdamW test")
    plt.plot(xsP, teP, marker="o", linestyle="--", label="PenaltyAdamW test")
    plt.xscale("log")
    plt.xlabel("steps (log)")
    plt.ylabel("test bit accuracy (%)")
    plt.title(f"{task_name}: test accuracy (N={N_train})")
    plt.legend()
    png = results_dir / f"{task_name}_testcurve_N{N_train}.png"
    pdf = results_dir / f"{task_name}_testcurve_N{N_train}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)

    print("Saved:", logA.name, logP.name)
    return logA, logP, png, pdf

In [19]:
MAJ_DATASETS = {
    "m5k1": {"dat": MK_DIR / "m5k1.dat", "N": 128},
    "m5k2": {"dat": MK_DIR / "m5k2.dat", "N": 256},
    "m5k3": {"dat": MK_DIR / "m5k3.dat", "N": 384},
    "m5k4": {"dat": MK_DIR / "m5k4.dat", "N": 512},
}
for k, cfg in MAJ_DATASETS.items():
    assert cfg["dat"].exists(), cfg["dat"]

TASK_DIR = OUT / "random_majority_mk"
TASK_DIR.mkdir(parents=True, exist_ok=True)

for k, cfg in MAJ_DATASETS.items():
    X, Y, IN, OUTD = load_dat_bits(cfg["dat"])
    assert IN == 32 and OUTD == 32 and len(X) == 10000
    N = cfg["N"]

    print(f"\n========== RandomMajority {k} | N={N} ==========")
    run_adamw_and_penalty(
        task_name=f"randmaj_{k}",
        X_all=X, Y_all=Y,
        make_model_fn=lambda: MLP32_depth5(),
        N_train=N,
        max_steps=1_000_000,
        results_dir=TASK_DIR,
        seed=0
    )

print("\nDONE random majority. Outputs:", TASK_DIR)


========== RandomMajority m5k1 | N=128 ==========
[randmaj_m5k1 | AdamW | N=128] step=       0 train= 49.73% test= 50.07% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       1 train= 49.73% test= 50.15% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       2 train= 49.83% test= 50.08% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       3 train= 49.83% test= 50.08% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       4 train= 49.83% test= 50.08% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       5 train= 49.83% test= 50.09% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       7 train= 50.00% test= 50.16% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       9 train= 50.34% test= 50.24% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=      12 train= 50.71% test= 50.40% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=      17 train= 51.51% test= 50.49% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=      23 trai

In [20]:
def snapshot_weights_per_layer(model: nn.Module):
    """Return list of flattened weight arrays per Linear layer."""
    snaps = []
    for m in model.modules():
        if isinstance(m, nn.Linear):
            snaps.append(m.weight.detach().cpu().flatten().numpy().copy())
    return snaps

def save_weight_hists(task_name: str, mode: str, N_train: int, step: int, snaps, results_dir: Path):
    """Save histograms for layer0 + last layer at a given checkpoint."""
    if len(snaps) == 0:
        return
    w0 = snaps[0]
    wL = snaps[-1]

    # layer 0
    fig = plt.figure()
    plt.hist(w0, bins=80)
    plt.title(f"{task_name} | {mode} | N={N_train} | step={step} | layer0")
    p = results_dir / f"{task_name}_{mode}_N{N_train}_step{step}_hist_layer0.png"
    fig.savefig(p, dpi=250, bbox_inches="tight")
    plt.close(fig)

    # last layer
    fig = plt.figure()
    plt.hist(wL, bins=80)
    plt.title(f"{task_name} | {mode} | N={N_train} | step={step} | lastlayer")
    p = results_dir / f"{task_name}_{mode}_N{N_train}_step{step}_hist_last.png"
    fig.savefig(p, dpi=250, bbox_inches="tight")
    plt.close(fig)

def run_adamw_and_penalty_with_figs(
    task_name: str,
    X_all: torch.Tensor,
    Y_all: torch.Tensor,
    make_model_fn,
    N_train: int,
    max_steps: int,
    results_dir: Path,
    seed: int = 0,
    lr: float = 1e-3,
    wd: float = 1e-4,
    lam_norm: float = 1e-2,
    lam_l1: float = 1e-4,
    log_points: int = 45,
    # where to take histogram snapshots (choose early/mid/late)
    hist_steps=(0, 1000, 10000, None),  # None = final
):
    results_dir.mkdir(parents=True, exist_ok=True)
    LOG_STEPS = log_spaced_steps(max_steps, log_points)
    torch.manual_seed(seed)

    Xtr = X_all[:N_train].to(device); Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device); Yte = Y_all[N_train:].to(device)

    # finalize histogram checkpoints
    hs = [s for s in hist_steps if s is not None]
    hs = set([s for s in hs if 0 <= s <= max_steps] + [max_steps])

    def train(mode: str):
        model = make_model_fn().to(device)
        opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

        tag = f"{task_name}_{mode}_N{N_train}_seed{seed}_steps{max_steps}"
        log_path = results_dir / f"{tag}.log"

        with open(log_path, "w") as f:
            f.write("# step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

        xs, tr_bits, te_bits, tr_exs, te_exs = [], [], [], [], []

        @torch.no_grad()
        def eval_log(step: int):
            model.eval()
            tr_logits = model(Xtr); te_logits = model(Xte)

            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()

            tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr)
            te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte)

            with open(log_path, "a") as f:
                f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_ex:.4f},{te_ex:.4f}\n")

            print(f"[{task_name} | {mode} | N={N_train}] step={step:>8d} "
                  f"train={tr_bit:6.2f}% test={te_bit:6.2f}% exact={tr_ex:5.1f}/{te_ex:5.1f}")

            xs.append(step); tr_bits.append(tr_bit); te_bits.append(te_bit)
            tr_exs.append(tr_ex); te_exs.append(te_ex)

            # save histograms at checkpoints
            if step in hs:
                snaps = snapshot_weights_per_layer(model)
                save_weight_hists(task_name, mode, N_train, step, snaps, results_dir)

            model.train()

        # step 0
        eval_log(0)
        model.train()

        for step in range(1, max_steps + 1):
            opt.zero_grad(set_to_none=True)
            logits = model(Xtr)
            base = loss_fn(logits, Ytr)

            if mode == "PenaltyAdamW":
                pen = 0.0
                for m in model.modules():
                    if isinstance(m, nn.Linear):
                        fan_in = m.weight.shape[1]
                        pen += lam_norm * norm_penalty_rows(m.weight, float(fan_in))
                        pen += lam_l1 * l1_penalty(m.weight)
                loss = base + pen
            else:
                loss = base

            loss.backward()
            opt.step()

            if step in LOG_STEPS:
                eval_log(step)

        return (np.array(xs), np.array(tr_bits), np.array(te_bits),
                np.array(tr_exs), np.array(te_exs), log_path)

    # ---- run both ----
    xsA, trA, teA, trExA, teExA, logA = train("AdamW")
    xsP, trP, teP, trExP, teExP, logP = train("PenaltyAdamW")

    # ---- save curves: test + train (bitwise) ----
    fig = plt.figure()
    plt.plot(xsA, teA, marker="o", label="AdamW test-bit")
    plt.plot(xsA, trA, marker="o", linestyle=":", label="AdamW train-bit")
    plt.plot(xsP, teP, marker="o", linestyle="--", label="Penalty test-bit")
    plt.plot(xsP, trP, marker="o", linestyle="dashdot", label="Penalty train-bit")
    plt.xscale("log")
    plt.xlabel("steps (log)")
    plt.ylabel("bitwise accuracy (%)")
    plt.title(f"{task_name}: train/test bit-acc (N={N_train})")
    plt.legend()
    png = results_dir / f"{task_name}_train_test_bitcurve_N{N_train}.png"
    pdf = results_dir / f"{task_name}_train_test_bitcurve_N{N_train}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)

    # ---- save exact curves ----
    fig = plt.figure()
    plt.plot(xsA, teExA, marker="o", label="AdamW test-exact")
    plt.plot(xsA, trExA, marker="o", linestyle=":", label="AdamW train-exact")
    plt.plot(xsP, teExP, marker="o", linestyle="--", label="Penalty test-exact")
    plt.plot(xsP, trExP, marker="o", linestyle="dashdot", label="Penalty train-exact")
    plt.xscale("log")
    plt.xlabel("steps (log)")
    plt.ylabel("exact accuracy (%)")
    plt.title(f"{task_name}: train/test exact-acc (N={N_train})")
    plt.legend()
    png = results_dir / f"{task_name}_train_test_exactcurve_N{N_train}.png"
    pdf = results_dir / f"{task_name}_train_test_exactcurve_N{N_train}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.close(fig)

    print("Saved logs:", logA.name, logP.name)
    return logA, logP, png, pdf

In [21]:
MAJ_DATASETS = {
    "m5k1": {"dat": MK_DIR / "m5k1.dat", "N": 128},
    "m5k2": {"dat": MK_DIR / "m5k2.dat", "N": 256},
    "m5k3": {"dat": MK_DIR / "m5k3.dat", "N": 384},
    "m5k4": {"dat": MK_DIR / "m5k4.dat", "N": 512},
}
for k, cfg in MAJ_DATASETS.items():
    assert cfg["dat"].exists(), cfg["dat"]

TASK_DIR = OUT / "random_majority_mk"
TASK_DIR.mkdir(parents=True, exist_ok=True)

for k, cfg in MAJ_DATASETS.items():
    X, Y, IN, OUTD = load_dat_bits(cfg["dat"])
    assert IN == 32 and OUTD == 32 and len(X) == 10000
    N = cfg["N"]

    print(f"\n========== RandomMajority {k} | N={N} ==========")
    run_adamw_and_penalty_with_figs(
        task_name=f"randmaj_{k}",
        X_all=X, Y_all=Y,
        make_model_fn=lambda: MLP32_depth5(),
        N_train=N,
        max_steps=1_000_000,
        results_dir=TASK_DIR,
        seed=0,
        log_points=45,
        hist_steps=(0, 1000, 10000, 100000, 1000000),
    )

print("\nDONE mk series. Outputs:", TASK_DIR)


========== RandomMajority m5k1 | N=128 ==========
[randmaj_m5k1 | AdamW | N=128] step=       0 train= 49.73% test= 50.07% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       1 train= 49.73% test= 50.15% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       2 train= 49.83% test= 50.08% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       3 train= 49.83% test= 50.08% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       4 train= 49.83% test= 50.08% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       5 train= 49.83% test= 50.09% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       7 train= 50.00% test= 50.16% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=       9 train= 50.34% test= 50.24% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=      12 train= 50.71% test= 50.40% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=      17 train= 51.51% test= 50.49% exact=  0.0/  0.0
[randmaj_m5k1 | AdamW | N=128] step=      23 trai

In [ ]:
## 30_1

In [22]:
# =========================
# CELL 8 — Rule-30 (1-step): AdamW vs PenaltyAdamW
# N = 64..512, steps=1e6
# Saves logs + curves + histograms
# =========================

DAT_30_1 = REPO / "30_1.dat"
assert DAT_30_1.exists(), f"Missing {DAT_30_1} (generate it with data_cellauto if needed)."

X1, Y1, IN, OUTD = load_dat_bits(DAT_30_1)
assert IN == 16 and OUTD == 16 and len(X1) == 10000

N_LIST = [64, 96, 128, 160, 256, 384, 512]
TASK_DIR = OUT / "rule30_1step"
TASK_DIR.mkdir(parents=True, exist_ok=True)

for N in N_LIST:
    print(f"\n========== Rule30 1-step | N={N} ==========")
    run_adamw_and_penalty_with_figs(
        task_name="rule30_1step",
        X_all=X1, Y_all=Y1,
        make_model_fn=lambda: MLP16_32_32_16(),
        N_train=N,
        max_steps=1_000_000,
        results_dir=TASK_DIR,
        seed=0,
        log_points=45,
        hist_steps=(0, 1000, 10000, 100000, 1000000),
    )

print("\nDONE rule30 1-step. Outputs:", TASK_DIR)


========== Rule30 1-step | N=64 ==========
[rule30_1step | AdamW | N=64] step=       0 train= 50.29% test= 49.49% exact=  0.0/  0.0
[rule30_1step | AdamW | N=64] step=       1 train= 50.88% test= 49.50% exact=  0.0/  0.0
[rule30_1step | AdamW | N=64] step=       2 train= 51.07% test= 49.51% exact=  0.0/  0.0
[rule30_1step | AdamW | N=64] step=       3 train= 50.88% test= 49.51% exact=  0.0/  0.0
[rule30_1step | AdamW | N=64] step=       4 train= 50.78% test= 49.52% exact=  0.0/  0.0
[rule30_1step | AdamW | N=64] step=       5 train= 51.46% test= 49.54% exact=  0.0/  0.0
[rule30_1step | AdamW | N=64] step=       7 train= 52.25% test= 49.61% exact=  0.0/  0.0
[rule30_1step | AdamW | N=64] step=       9 train= 52.54% test= 49.81% exact=  0.0/  0.0
[rule30_1step | AdamW | N=64] step=      12 train= 53.52% test= 50.05% exact=  0.0/  0.0
[rule30_1step | AdamW | N=64] step=      17 train= 54.00% test= 50.47% exact=  0.0/  0.0
[rule30_1step | AdamW | N=64] step=      23 train= 55.37% test= 50

In [23]:
# =========================
# CELL 9 — Rule-30 (2-step): AdamW vs PenaltyAdamW
# N = 64..512, steps=2e6
# Saves logs + curves + histograms
# =========================

DAT_30_2 = REPO / "30_2.dat"
assert DAT_30_2.exists(), f"Missing {DAT_30_2}"

X2, Y2, IN, OUTD = load_dat_bits(DAT_30_2)
assert IN == 16 and OUTD == 16 and len(X2) == 10000

N_LIST = [64, 96, 128, 160, 256, 384, 512]
TASK_DIR = OUT / "rule30_2step"
TASK_DIR.mkdir(parents=True, exist_ok=True)

for N in N_LIST:
    print(f"\n========== Rule30 2-step | N={N} ==========")
    run_adamw_and_penalty_with_figs(
        task_name="rule30_2step",
        X_all=X2, Y_all=Y2,
        make_model_fn=lambda: MLP16_32_32_16(),
        N_train=N,
        max_steps=2_000_000,
        results_dir=TASK_DIR,
        seed=0,
        log_points=55,  # more dots for longer run
        hist_steps=(0, 1000, 10000, 100000, 2000000),
    )

print("\nDONE rule30 2-step. Outputs:", TASK_DIR)


========== Rule30 2-step | N=64 ==========
[rule30_2step | AdamW | N=64] step=       0 train= 51.56% test= 49.92% exact=  0.0/  0.0
[rule30_2step | AdamW | N=64] step=       1 train= 51.66% test= 50.02% exact=  0.0/  0.0
[rule30_2step | AdamW | N=64] step=       2 train= 52.34% test= 50.11% exact=  0.0/  0.0
[rule30_2step | AdamW | N=64] step=       3 train= 52.34% test= 50.23% exact=  0.0/  0.0
[rule30_2step | AdamW | N=64] step=       4 train= 53.32% test= 50.30% exact=  0.0/  0.0
[rule30_2step | AdamW | N=64] step=       5 train= 53.71% test= 50.44% exact=  0.0/  0.0
[rule30_2step | AdamW | N=64] step=       7 train= 54.39% test= 50.54% exact=  0.0/  0.0
[rule30_2step | AdamW | N=64] step=       9 train= 55.37% test= 50.60% exact=  0.0/  0.0
[rule30_2step | AdamW | N=64] step=      11 train= 56.93% test= 50.61% exact=  0.0/  0.0
[rule30_2step | AdamW | N=64] step=      15 train= 58.30% test= 50.53% exact=  0.0/  0.0
[rule30_2step | AdamW | N=64] step=      19 train= 58.40% test= 50

In [35]:
# =========================
# CELL 10 — Write LaTeX that includes all saved PDFs under OUT/
# =========================

tex_dir = OUT / "latex_bundle"
tex_dir.mkdir(parents=True, exist_ok=True)

# Collect PDFs, excluding anything generated in latex_bundle/
pdfs = sorted([
    p for p in OUT.rglob("*.pdf")
    if p.is_file() and tex_dir not in p.parents
])

print("Found PDFs:", len(pdfs))

tex_path = tex_dir / "all_figs.tex"
lines = []

# Preamble
lines += [r"\documentclass[11pt]{article}"]
lines += [r"\usepackage[margin=1in]{geometry}"]
lines += [r"\usepackage{graphicx}"]
lines += [r"\usepackage{float}"]
lines += [r"\begin{document}"]

lines += [r"\section*{AdamW vs PenaltyAdamW: mk + rule30 (1-step / 2-step)}"]
lines += [r"\noindent Auto-generated from saved PDF figures."]

# Figures
for i, p in enumerate(pdfs, start=1):
    rel = p.relative_to(OUT).as_posix()

    lines += [r"\begin{figure}[H]\centering"]
    lines += [rf"\includegraphics[width=\linewidth]{{\detokenize{{../{rel}}}}}"]
    lines += [rf"\caption{{Figure {i}: \detokenize{{{p.name}}}}}"]
    lines += [r"\end{figure}\clearpage"]

lines += [r"\end{document}"]

tex_path.write_text("\n".join(lines))

print("Wrote:", tex_path)
print("\nCompile with:")
print(f"  cd {tex_dir}")
print("  pdflatex all_figs.tex")

Found PDFs: 40
Wrote: /Users/mkrishan/Documents/boolearn-main/paper_runs_adamw_vs_penalty/latex_bundle/all_figs.tex

Compile with:
  cd /Users/mkrishan/Documents/boolearn-main/paper_runs_adamw_vs_penalty/latex_bundle
  pdflatex all_figs.tex


In [3]:
# =========================
# CELL 1 — Shared setup (AdamW + PenaltyAdamW), log-spaced printing + logging
# =========================

from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ---- EDIT PATHS ----
REPO = Path("/Users/mkrishan/Documents/boolearn-main")
assert REPO.exists(), REPO

DAT_30_1 = REPO / "30_1.dat"
DAT_30_2 = REPO / "30_2.dat"
assert DAT_30_1.exists(), DAT_30_1
assert DAT_30_2.exists(), DAT_30_2

OUTDIR = REPO / "results_rule30_new_arch_adamw_vs_penalty"
OUTDIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

loss_fn = nn.BCEWithLogitsLoss()

def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    it = iter(lines)
    N_total = int(next(it))
    _, IN = map(int, next(it).split())
    _, OUT = map(int, next(it).split())
    X, Y = [], []
    for _ in range(N_total):
        X.append(list(map(int, next(it).split())))
        Y.append(list(map(int, next(it).split())))
    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)
    return X, Y, IN, OUT

@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred  = (probs >= thresh).float()
    bit   = (pred == y_true).float().mean().item() * 100.0
    exact = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit, exact

def log_spaced_steps(max_steps: int, n_points: int = 55):
    # logspace from 1..max_steps plus 0
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return set([0] + steps.tolist())

# --- penalties (same as your working penalty AdamW code) ---
def norm_penalty_rows(W: torch.Tensor, target_norm2: float):
    row_norm2 = (W * W).sum(dim=1)
    return ((row_norm2 - target_norm2) ** 2).mean()

def l1_penalty(W: torch.Tensor):
    return W.abs().mean()

def penalty_term(model: nn.Module, lam_norm: float, lam_l1: float):
    pen = 0.0
    for mod in model.modules():
        if isinstance(mod, nn.Linear):
            fan_in = mod.weight.shape[1]
            pen = pen + lam_norm * norm_penalty_rows(mod.weight, target_norm2=float(fan_in))
            pen = pen + lam_l1 * l1_penalty(mod.weight)
    return pen

device: cpu


In [4]:
# =========================
# CELL 2 — Architectures
# =========================

class MLP16_48_16(nn.Module):
    """16 -> 48 -> 16"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(16, 48), nn.ReLU(inplace=True),
            nn.Linear(48, 16),
        )
    def forward(self, x):
        return self.net(x)

class MLP16_48_48_48_16(nn.Module):
    """16 -> 48 -> 48 -> 48 -> 16"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(16, 48), nn.ReLU(inplace=True),
            nn.Linear(48, 48), nn.ReLU(inplace=True),
            nn.Linear(48, 48), nn.ReLU(inplace=True),
            nn.Linear(48, 16),
        )
    def forward(self, x):
        return self.net(x)

In [5]:
# =========================
# CELL 3 — Training runner (full-batch, deterministic, log-spaced)
# Writes .log and prints checkpoints
# =========================

def run_one_experiment(
    *, name: str,
    X_all: torch.Tensor, Y_all: torch.Tensor,
    N_train: int,
    make_model_fn,
    max_steps: int,
    seed: int = 0,
    lr: float = 1e-3,
    wd: float = 1e-4,
    lam_norm: float = 1e-2,
    lam_l1: float = 1e-4,
    n_log_points: int = 55,
):
    """
    Runs BOTH:
      - AdamW
      - PenaltyAdamW (AdamW + norm/L1 penalties)
    Saves:
      <name>_AdamW_....log
      <name>_PenaltyAdamW_....log
    """

    LOG_STEPS = log_spaced_steps(max_steps, n_points=n_log_points)

    def train_mode(mode: str):
        assert mode in ["AdamW", "PenaltyAdamW"]
        tag = f"{name}_{mode}_N{N_train}_seed{seed}_steps{max_steps}"
        log_path = OUTDIR / f"{tag}.log"

        if log_path.exists():
            print("[SKIP]", log_path.name)
            return log_path

        torch.manual_seed(seed)

        Xtr = X_all[:N_train].to(device);  Ytr = Y_all[:N_train].to(device)
        Xte = X_all[N_train:].to(device);  Yte = Y_all[N_train:].to(device)

        model = make_model_fn().to(device)
        opt   = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

        with open(log_path, "w") as f:
            f.write("# step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")
            f.write(f"# mode={mode}, lr={lr}, wd={wd}, lam_norm={lam_norm}, lam_l1={lam_l1}\n")

        @torch.no_grad()
        def eval_and_log(step: int):
            model.eval()
            tr_logits = model(Xtr); te_logits = model(Xte)

            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()
            tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr, 0.5)
            te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte, 0.5)

            with open(log_path, "a") as f:
                f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_ex:.4f},{te_ex:.4f}\n")

            print(f"[{name} | {mode} | N={N_train}] step={step:>8d} "
                  f"train={tr_bit:6.2f}% test={te_bit:6.2f}% exact={tr_ex:5.1f}/{te_ex:5.1f}")

            model.train()

        # step 0
        eval_and_log(0)

        model.train()
        for step in range(1, max_steps + 1):
            opt.zero_grad(set_to_none=True)
            logits = model(Xtr)
            base = loss_fn(logits, Ytr)

            if mode == "PenaltyAdamW":
                loss = base + penalty_term(model, lam_norm=lam_norm, lam_l1=lam_l1)
            else:
                loss = base

            loss.backward()
            opt.step()

            if step in LOG_STEPS:
                eval_and_log(step)

        print("Saved ->", log_path)
        return log_path

    print(f"\n========== {name} | N={N_train} | steps={max_steps} ==========")
    p1 = train_mode("AdamW")
    p2 = train_mode("PenaltyAdamW")
    return p1, p2

In [10]:
# =========================
# CELL 4 — Run BOTH tasks with requested architectures + step budgets
# Rule-30 (1-step): arch 16->48->16, steps=1e6
# Rule-30 (2-step): arch 16->48->48->48->16, steps=2e6
# Both: AdamW vs PenaltyAdamW
# =========================

SEED = 0
LR   = 1e-3
WD   = 1e-4

# penalty hyperparams (match your working rule30 penalty setup)
LAM_NORM = 1e-2
LAM_L1   = 1e-4

# --- load datasets ---
X1, Y1, IN1, OUT1 = load_dat_bits(DAT_30_1)
X2, Y2, IN2, OUT2 = load_dat_bits(DAT_30_2)

assert IN1 == 16 and OUT1 == 16 and len(X1) == 10000
assert IN2 == 16 and OUT2 == 16 and len(X2) == 10000

# --- run lists ---
N_LIST_RULE30_1 = [64, 96, 128, 160, 256, 384, 512]     # you said safe to include wide range
N_LIST_RULE30_2 = [128, 256, 384, 512]                  # typical for 2-step

# --- rule-30 (1-step): 16->48->16, steps=1e6 ---
for N in N_LIST_RULE30_1:
    run_one_experiment(
        name="rule30_1step_mlp16_48_16",
        X_all=X1, Y_all=Y1,
        N_train=N,
        make_model_fn=MLP16_48_16,
        max_steps=1_000_000,
        seed=SEED, lr=LR, wd=WD,
        lam_norm=LAM_NORM, lam_l1=LAM_L1,
        n_log_points=55,
    )

# --- rule-30 (2-step): 16->48->48->48->16, steps=2e6 ---
for N in N_LIST_RULE30_2:
    run_one_experiment(
        name="rule30_2step_mlp16_48_48_48_16",
        X_all=X2, Y_all=Y2,
        N_train=N,
        make_model_fn=MLP16_48_48_48_16,
        max_steps=2_000_000,
        seed=SEED, lr=LR, wd=WD,
        lam_norm=LAM_NORM, lam_l1=LAM_L1,
        n_log_points=60,   # a few more points since longer run
    )

print("\nALL DONE. Logs in:", OUTDIR)


========== rule30_1step_mlp16_48_16 | N=64 | steps=1000000 ==========
[SKIP] rule30_1step_mlp16_48_16_AdamW_N64_seed0_steps1000000.log
[SKIP] rule30_1step_mlp16_48_16_PenaltyAdamW_N64_seed0_steps1000000.log

========== rule30_1step_mlp16_48_16 | N=96 | steps=1000000 ==========
[SKIP] rule30_1step_mlp16_48_16_AdamW_N96_seed0_steps1000000.log
[SKIP] rule30_1step_mlp16_48_16_PenaltyAdamW_N96_seed0_steps1000000.log

========== rule30_1step_mlp16_48_16 | N=128 | steps=1000000 ==========
[SKIP] rule30_1step_mlp16_48_16_AdamW_N128_seed0_steps1000000.log
[SKIP] rule30_1step_mlp16_48_16_PenaltyAdamW_N128_seed0_steps1000000.log

========== rule30_1step_mlp16_48_16 | N=160 | steps=1000000 ==========
[SKIP] rule30_1step_mlp16_48_16_AdamW_N160_seed0_steps1000000.log
[SKIP] rule30_1step_mlp16_48_16_PenaltyAdamW_N160_seed0_steps1000000.log

========== rule30_1step_mlp16_48_16 | N=256 | steps=1000000 ==========
[SKIP] rule30_1step_mlp16_48_16_AdamW_N256_seed0_steps1000000.log
[SKIP] rule30_1step_mlp1

In [12]:
# =========================
# MAKE ONE PDF OF ALL FIGURES (from saved .log files)
# - Reads OUTDIR/*.log
# - Plots train/test bit-acc + exact-acc
# - Writes a single multi-page PDF
# =========================

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Uses OUTDIR from your training notebook cell:
# OUTDIR = REPO / "results_rule30_new_arch_adamw_vs_penalty"
assert "OUTDIR" in globals(), "OUTDIR not defined. Run the training cells first."

log_files = sorted(Path(OUTDIR).glob("*.log"))
print("Found", len(log_files), "log files in", OUTDIR)

def read_log_csv(log_path: Path):
    """Read our .log format, skipping comment lines."""
    rows = []
    with open(log_path, "r") as f:
        for ln in f:
            ln = ln.strip()
            if not ln or ln.startswith("#"):
                continue
            parts = ln.split(",")
            if len(parts) < 7:
                continue
            step = int(float(parts[0]))
            tr_loss = float(parts[1])
            te_loss = float(parts[2])
            tr_bit  = float(parts[3])
            te_bit  = float(parts[4])
            tr_ex   = float(parts[5])
            te_ex   = float(parts[6])
            rows.append((step, tr_loss, te_loss, tr_bit, te_bit, tr_ex, te_ex))
    if not rows:
        return None
    arr = np.array(rows, dtype=float)
    return dict(
        step=arr[:,0],
        tr_loss=arr[:,1],
        te_loss=arr[:,2],
        tr_bit=arr[:,3],
        te_bit=arr[:,4],
        tr_ex=arr[:,5],
        te_ex=arr[:,6],
    )

pdf_path = Path(OUTDIR) / "all_figs_rule30_adamw_vs_penalty.pdf"

with PdfPages(pdf_path) as pdf:
    for lf in log_files:
        data = read_log_csv(lf)
        if data is None:
            print("[skip empty]", lf.name)
            continue

        steps = data["step"]

        # -------- Figure 1: Bitwise accuracy --------
        plt.figure(figsize=(7.5, 4.5))
        plt.plot(steps, data["tr_bit"], marker="o", markersize=3, linewidth=1.2, label="train bit-acc (%)")
        plt.plot(steps, data["te_bit"], marker="o", markersize=3, linewidth=1.2, label="test bit-acc (%)")
        plt.xscale("symlog", linthresh=1)  # shows 0 nicely and log afterwards
        plt.xlabel("Step")
        plt.ylabel("Bitwise accuracy (%)")
        plt.title(lf.stem + "  |  bitwise accuracy")
        plt.grid(True, which="both", linewidth=0.3)
        plt.legend()
        plt.tight_layout()
        pdf.savefig()
        plt.close()

        # -------- Figure 2: Exact reconstruction --------
        plt.figure(figsize=(7.5, 4.5))
        plt.plot(steps, data["tr_ex"], marker="o", markersize=3, linewidth=1.2, label="train exact (%)")
        plt.plot(steps, data["te_ex"], marker="o", markersize=3, linewidth=1.2, label="test exact (%)")
        plt.xscale("symlog", linthresh=1)
        plt.xlabel("Step")
        plt.ylabel("Exact match (%)")
        plt.title(lf.stem + "  |  exact reconstruction")
        plt.grid(True, which="both", linewidth=0.3)
        plt.legend()
        plt.tight_layout()
        pdf.savefig()
        plt.close()

        # -------- (Optional) Figure 3: Loss curves --------
        plt.figure(figsize=(7.5, 4.5))
        plt.plot(steps, data["tr_loss"], marker="o", markersize=3, linewidth=1.2, label="train loss")
        plt.plot(steps, data["te_loss"], marker="o", markersize=3, linewidth=1.2, label="test loss")
        plt.xscale("symlog", linthresh=1)
        plt.xlabel("Step")
        plt.ylabel("BCEWithLogitsLoss")
        plt.title(lf.stem + "  |  loss")
        plt.grid(True, which="both", linewidth=0.3)
        plt.legend()
        plt.tight_layout()
        pdf.savefig()
        plt.close()

print("Wrote:", pdf_path)

Found 22 log files in /Users/mkrishan/Documents/boolearn-main/results_rule30_new_arch_adamw_vs_penalty
Wrote: /Users/mkrishan/Documents/boolearn-main/results_rule30_new_arch_adamw_vs_penalty/all_figs_rule30_adamw_vs_penalty.pdf
